In [1]:
from pathlib import Path

edm = Path("../data/raw/edm")
for f in sorted(edm.rglob("*")):
    if f.is_file():
        print(f.relative_to(edm), "|", round(f.stat().st_size / 1e6, 1), "MB")

2024 Detailed EDM Start-Stop Data\Anglian 2024 Detailed EDM Data.xlsx | 7.3 MB
2024 Detailed EDM Start-Stop Data\Northumbrian 2024 Detailed EDM Data.csv | 8.7 MB
2024 Detailed EDM Start-Stop Data\Severn Trent 2024 Detailed EDM Data.xlsx | 7.4 MB
2024 Detailed EDM Start-Stop Data\South West 2024 Detailed EDM Data (1).xlsx | 7.6 MB
2024 Detailed EDM Start-Stop Data\Southern 2024 Detailed EDM Data.csv | 6.1 MB
2024 Detailed EDM Start-Stop Data\Thames 2024 Detailed EDM Data.csv | 0.9 MB
2024 Detailed EDM Start-Stop Data\United Utilities 2024 Detailed EDM Data.csv | 18.1 MB
2024 Detailed EDM Start-Stop Data\Welsh 2024 Detailed EDM Data.xlsx | 0.4 MB
2024 Detailed EDM Start-Stop Data\Wessex 2024 Detailed EDM Data.csv | 35.0 MB
2024 Detailed EDM Start-Stop Data\Yorkshire 2024 Detailed EDM Data.csv | 17.4 MB
2025 Detailed EDM Start-Stop Data\Anglian Water Detailed Data 2025.csv | 8.0 MB
2025 Detailed EDM Start-Stop Data\Northumbrian 2025 Detailed EDM Data.csv | 7.0 MB
2025 Detailed EDM Start-S

In [2]:
import pandas as pd

uu24_path = edm / "2024 Detailed EDM Start-Stop Data" / "United Utilities 2024 Detailed EDM Data.csv"
uu24 = pd.read_csv(uu24_path, nrows=5)
print(uu24.columns.tolist())
uu24

['Unique ID', 'Site Name', 'Discharge Start (GMT)', 'Discharge Stop (GMT)']


,Unique ID,Site Name,Discharge Start (GMT),Discharge Stop (GMT)
0,UUP02182,WRENBURY WWTW,2024/01/01T20:00:00.000Z,2024/01/01T22:15:00.000Z
1,UUP02182,WRENBURY WWTW,2024/01/02T11:45:00.000Z,2024/01/02T23:15:00.000Z
2,UUP02182,WRENBURY WWTW,2024/01/03T10:30:00.000Z,2024/01/03T15:00:00.000Z
3,UUP02182,WRENBURY WWTW,2024/02/06T14:15:00.000Z,2024/02/06T14:45:00.000Z
4,UUP02182,WRENBURY WWTW,2024/02/09T01:30:00.000Z,2024/02/09T02:30:00.000Z


In [3]:
uu24 = pd.read_csv(uu24_path)

fmt = "%Y/%m/%dT%H:%M:%S.%fZ"
uu24["start"] = pd.to_datetime(uu24["Discharge Start (GMT)"], format=fmt, utc=True, errors="coerce")
uu24["stop"] = pd.to_datetime(uu24["Discharge Stop (GMT)"], format=fmt, utc=True, errors="coerce")
uu24["duration_hours"] = (uu24["stop"] - uu24["start"]).dt.total_seconds() / 3600

print("Total spill events:", len(uu24))
print("Unique outlets:", uu24["Unique ID"].nunique())
print("Start dates that failed to read:", uu24["start"].isna().sum())
print("Stop dates that failed to read:", uu24["stop"].isna().sum())
print(uu24["duration_hours"].describe())

Total spill events: 216379
Unique outlets: 1981
Start dates that failed to read: 174
Stop dates that failed to read: 174
count    216205.000000
mean          2.077349
std          14.412948
min           0.000000
25%           0.076667
50%           0.266667
75%           0.966667
max        2905.000000
Name: duration_hours, dtype: float64


In [4]:
uu24[uu24["start"].isna()].head(10)

,Unique ID,Site Name,Discharge Start (GMT),Discharge Stop (GMT),start,stop,duration_hours
129865,UUP00724,FAZAKERLEY WWTW,2024/01/01T00:00:00Z,2024/01/01T00:10:49Z,NaT,NaT,NaN
129866,UUP00724,FAZAKERLEY WWTW,2024/01/01T20:23:26Z,2024/01/01T22:56:55Z,NaT,NaT,NaN
129867,UUP00724,FAZAKERLEY WWTW,2024/01/11T23:27:57Z,2024/01/11T23:28:00Z,NaT,NaT,NaN
129868,UUP00724,FAZAKERLEY WWTW,2024/01/14T18:09:57Z,2024/01/14T18:10:00Z,NaT,NaT,NaN
129869,UUP00724,FAZAKERLEY WWTW,2024/01/16T01:49:59Z,2024/01/16T01:50:02Z,NaT,NaT,NaN
129870,UUP00724,FAZAKERLEY WWTW,2024/01/16T01:51:58Z,2024/01/16T01:52:01Z,NaT,NaT,NaN
129871,UUP00724,FAZAKERLEY WWTW,2024/01/16T01:55:58Z,2024/01/16T01:56:01Z,NaT,NaT,NaN
129872,UUP00724,FAZAKERLEY WWTW,2024/01/16T02:01:58Z,2024/01/16T02:02:02Z,NaT,NaT,NaN
129873,UUP00724,FAZAKERLEY WWTW,2024/01/16T02:05:57Z,2024/01/16T02:06:00Z,NaT,NaT,NaN
129874,UUP00724,FAZAKERLEY WWTW,2024/01/16T04:49:58Z,2024/01/16T04:50:01Z,NaT,NaT,NaN


In [5]:
def parse_time(col):
    with_ms = pd.to_datetime(col, format="%Y/%m/%dT%H:%M:%S.%fZ", utc=True, errors="coerce")
    without_ms = pd.to_datetime(col, format="%Y/%m/%dT%H:%M:%SZ", utc=True, errors="coerce")
    return with_ms.fillna(without_ms)

uu24["start"] = parse_time(uu24["Discharge Start (GMT)"])
uu24["stop"] = parse_time(uu24["Discharge Stop (GMT)"])
uu24["duration_hours"] = (uu24["stop"] - uu24["start"]).dt.total_seconds() / 3600

print("Start failed:", uu24["start"].isna().sum())
print("Stop failed:", uu24["stop"].isna().sum())

Start failed: 0
Stop failed: 0


In [6]:
print("Zero-duration events:", (uu24["duration_hours"] == 0).sum())
print("Negative durations:", (uu24["duration_hours"] < 0).sum())
print("Longer than 7 days:", (uu24["duration_hours"] > 24 * 7).sum())

uu24.sort_values("duration_hours", ascending=False).head(10)

Zero-duration events: 37
Negative durations: 0
Longer than 7 days: 142


,Unique ID,Site Name,Discharge Start (GMT),Discharge Stop (GMT),start,stop,duration_hours
74595,UUP00803,GLEASTON CASTLE CSO,2024/01/01T00:00:00.000Z,2024/05/01T01:00:00.000Z,2024-01-01 00:00:00+00:00,2024-05-01 01:00:00+00:00,2905.000000
216041,UUP00375,BURTON ROAD WITHINGTON CSO 28056,2024/10/06T12:42:57.000Z,2024/12/31T23:59:59.000Z,2024-10-06 12:42:57+00:00,2024-12-31 23:59:59+00:00,2075.283889
6188,UUP01880,STRANGEWAY NO.1 WW NTWK PS,2024/01/05T08:57:18.000Z,2024/03/10T09:32:01.000Z,2024-01-05 08:57:18+00:00,2024-03-10 09:32:01+00:00,1560.578611
97408,UUP02152,WINTON PUMPING STATION,2024/01/21T10:12:51.000Z,2024/03/12T00:53:26.000Z,2024-01-21 10:12:51+00:00,2024-03-12 00:53:26+00:00,1214.676389
6189,UUP01880,STRANGEWAY NO.1 WW NTWK PS,2024/03/10T09:43:31.000Z,2024/04/27T20:22:15.000Z,2024-03-10 09:43:31+00:00,2024-04-27 20:22:15+00:00,1162.645556
101583,UUP01022,INGLETON STW,2024/01/20T09:05:38.000Z,2024/03/08T02:56:21.000Z,2024-01-20 09:05:38+00:00,2024-03-08 02:56:21+00:00,1145.845278
118090,UUP00417,CARTMEL IN CARK PUMPING STATION,2024/01/04T21:46:28.000Z,2024/02/19T09:37:27.000Z,2024-01-04 21:46:28+00:00,2024-02-19 09:37:27+00:00,1091.849722
147725,UUP00109,ALTON STREET NO 2 CSO 161BP,2024/01/01T00:00:00.000Z,2024/02/09T10:02:00.000Z,2024-01-01 00:00:00+00:00,2024-02-09 10:02:00+00:00,946.033333
118091,UUP00417,CARTMEL IN CARK PUMPING STATION,2024/02/19T09:38:34.000Z,2024/03/23T18:07:00.000Z,2024-02-19 09:38:34+00:00,2024-03-23 18:07:00+00:00,800.473889
118178,UUP00417,CARTMEL IN CARK PUMPING STATION,2024/07/27T05:44:17.000Z,2024/08/28T10:38:21.000Z,2024-07-27 05:44:17+00:00,2024-08-28 10:38:21+00:00,772.901111


In [8]:
year_start = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
year_end = pd.Timestamp("2024-12-31 23:59:59", tz="UTC")

uu24["is_zero"] = uu24["duration_hours"] == 0
uu24["is_long"] = uu24["duration_hours"] > 24 * 7
uu24["cut_at_year_edge"] = (uu24["start"] == year_start) | (uu24["stop"] >= year_end)

print("Zero:", uu24["is_zero"].sum())
print("Long (>7 days):", uu24["is_long"].sum())
print("Cut at year edge:", uu24["cut_at_year_edge"].sum())

Zero: 37
Long (>7 days): 142
Cut at year edge: 415


## Data quality findings: United Utilities 2024 EDM data
- 216,379 spill events from 1,981 outlets.
- 174 timestamps had no milliseconds; fixed by parsing both formats.
- 37 zero-duration events: likely sensor noise.
- 142 events lasted over 7 days: possibly real (groundwater) or stuck sensors.
- 415 events are cut at the year boundary; they must be joined with 2023/2025 data.
- Flagged rows are under 0.3% of the data, so the dataset is mostly clean.

In [9]:
uu25_path = edm / "2025 Detailed EDM Start-Stop Data" / "United Utilities 2025 Detailed EDM Data.csv"
uu26_path = edm / "2026 Year to Sept Start-Stop Data" / "United Utilities Detailed Data 2026 (1).xlsx"

uu25_peek = pd.read_csv(uu25_path, nrows=3)
uu26_peek = pd.read_excel(uu26_path, nrows=3)

print("2025 columns:", uu25_peek.columns.tolist())
print("2026 columns:", uu26_peek.columns.tolist())
display(uu25_peek)
display(uu26_peek)

2025 columns: ['Unique ID', 'Site Name', 'Discharge Start (GMT)', 'Discharge Stop (GMT)', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14']
2026 columns: ['Unique ID', 'Site Name (EA consent database)', 'Discharge Start (GMT)', 'Discharge Stop (GMT)']


,Unique ID,Site Name,Discharge Start (GMT),Discharge Stop (GMT),Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,UUP00002,10 WARWICK AVENUE CSO (42058),2025-01-01T00:00:00.000Z,2025-01-01T03:30:00.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,UUP00002,10 WARWICK AVENUE CSO (42058),2025-01-01T04:05:00.000Z,2025-01-01T05:00:00.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,UUP00002,10 WARWICK AVENUE CSO (42058),2025-01-01T06:45:00.000Z,2025-01-01T06:50:00.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Unique ID,Site Name (EA consent database),Discharge Start (GMT),Discharge Stop (GMT)
0,UUP00002,10 WARWICK AVENUE CSO (42058),2026-04-15T17:04:00Z,2026-04-15T17:34:00Z
1,UUP00003,102 MIDDLEWICH ROAD CSO,2026-04-03T16:58:00Z,2026-04-03T17:18:00Z
2,UUP00003,102 MIDDLEWICH ROAD CSO,2026-04-11T19:16:00Z,2026-04-11T19:36:00Z


In [12]:
def load_uu(path, year):
    if path.suffix == ".csv":
        df = pd.read_csv(path).iloc[:, :4]
        df.columns = ["outlet_id", "site_name", "start_raw", "stop_raw"]
    else:
        sheets = pd.read_excel(path, sheet_name=None)
        parts = []
        for name, sheet in sheets.items():
            sheet = sheet.iloc[:, :4]
            sheet.columns = ["outlet_id", "site_name", "start_raw", "stop_raw"]
            parts.append(sheet)
        df = pd.concat(parts, ignore_index=True)

    for col in ["start", "stop"]:
        text = df[f"{col}_raw"].astype(str).str.replace("/", "-")
        df[col] = pd.to_datetime(text, format="ISO8601", utc=True, errors="coerce")

    df["duration_hours"] = (df["stop"] - df["start"]).dt.total_seconds() / 3600
    df["file_year"] = year
    return df

spills = pd.concat([
    load_uu(uu24_path, 2024),
    load_uu(uu25_path, 2025),
    load_uu(uu26_path, 2026),
], ignore_index=True)

print(spills.groupby("file_year").size())
print("Start failed:", spills["start"].isna().sum())
print("Stop failed:", spills["stop"].isna().sum())

file_year
2024    216379
2025    175358
2026     73634
dtype: int64
Start failed: 0
Stop failed: 0


### Investigation: 2026 row count looked too low
The first load gave only 3,802 rows for Jan–Sep 2026, versus 175,358 for 2025, and the data started in April.
Checking the Excel file showed 9 monthly sheets, with April first; `read_excel` reads only the first sheet by default.
**Fix:** `load_uu` now reads every sheet and combines them, giving 73,634 rows for 2026.

In [11]:
sheets = pd.read_excel(uu26_path, sheet_name=None)
for name, df in sheets.items():
    print(name, "|", len(df), "rows")

April | 3802 rows
February | 14213 rows
January | 8995 rows
March | 15094 rows
May | 4691 rows
June | 7551 rows
July | 595 rows
August | 9731 rows
September | 8962 rows


In [13]:
out_path = Path("../data/processed/uu_spills_2024_2026.parquet")
spills.to_parquet(out_path, index=False)

print("Saved", len(spills), "rows")
print("File size:", round(out_path.stat().st_size / 1e6, 1), "MB")

Saved 465371 rows
File size: 15.1 MB


## Day 2 summary
- Fetched daily Manchester rainfall 2020–2026 from the Open-Meteo API.
- Loaded United Utilities EDM spill data for 2024, 2025 and Jan–Sep 2026: 465,371 spill events.
- Handled three different file formats: CSV vs Excel, changing column names, and two date formats.
- Found and fixed a hidden issue: the 2026 Excel file had 9 monthly sheets, and only the first was being read.
- Saved the combined clean dataset as Parquet (15.1 MB, about 60% smaller than the raw files).
- **Next:** get outlet locations (latitude/longitude) so that spills can be matched to rainfall.